# Phase 4: SQL Business Analysis

## 1. Introduction
This phase uses SQL to analyze Netflix customer, subscription, viewing, payment, support, and feedback data. The purpose is to answer important business questions and identify patterns that can help improve customer retention and business performance.

## 2. Objectives
- Analyze customer and subscription trends.
- Understand customer churn patterns.
- Examine payment failures and their relationship with churn.
- Analyze viewing activity and customer engagement.
- Evaluate customer feedback and support performance.
- Identify actionable business insights.

## 3. Database Setup and Dataset Overview
The nine cleaned datasets were loaded into an SQLite database using Python and Google Colab. SQL queries are used to perform business analysis across the datasets.

## 4. SQL Business Analysis
Each business question includes:
- Business question
- SQL query
- Query output
- Key findings
- Business interpretation

## 5. Overall Key Findings
*To be completed after executing and reviewing the SQL queries.*

## 6. Business Recommendations
*Recommendations will be based on the actual SQL results.*

## 7. Conclusion
*Final conclusions will be added after completing the SQL analysis.*

In [6]:
import pandas as pd
import sqlite3
import os

# Connect to SQLite database in Google Colab
conn = sqlite3.connect("/content/netflix_customer_intelligence.db")

# Load the cleaned datasets
cleaned_datasets = {
    "customers": pd.read_csv("/content/customers.csv"),
    "subscriptions": pd.read_csv("/content/subscriptions.csv"),
    "subscription_plans": pd.read_csv("/content/subscription_plans.csv"),
    "viewing_activity": pd.read_csv("/content/viewing_activity.csv"),
    "support_tickets": pd.read_csv("/content/support_tickets.csv"),
    "payments": pd.read_csv("/content/payments.csv"),
    "churn_labels": pd.read_csv("/content/churn_labels.csv"),
    "customer_feedback": pd.read_csv("/content/customer_feedback.csv"),
    "content": pd.read_csv("/content/content.csv")
}

# Save all datasets as SQL tables
for table_name, df in cleaned_datasets.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)

print("All 9 datasets successfully loaded into SQLite!")

# Verify table names and record counts
for table_name in cleaned_datasets:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_records FROM {table_name}", conn
    ).iloc[0, 0]
    print(f"{table_name}: {count:,} records")

All 9 datasets successfully loaded into SQLite!
customers: 8,000 records
subscriptions: 8,875 records
subscription_plans: 3 records
viewing_activity: 66,422 records
support_tickets: 6,120 records
payments: 110,735 records
churn_labels: 8,000 records
customer_feedback: 5,046 records
content: 500 records


In [7]:
tables = [
    "customers", "subscriptions", "subscription_plans",
    "viewing_activity", "support_tickets", "payments",
    "churn_labels", "customer_feedback", "content"
]

for table in tables:
    print(f"\n{'='*55}")
    print(f"TABLE: {table.upper()}")
    print("="*55)

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table})", conn
    )
    print(schema[["name", "type"]].to_string(index=False))


TABLE: CUSTOMERS
               name type
        customer_id TEXT
         first_name TEXT
          last_name TEXT
                age REAL
             gender TEXT
            country TEXT
               city TEXT
  registration_date TEXT
acquisition_channel TEXT
   customer_segment TEXT
 preferred_language TEXT
age_validation_flag TEXT

TABLE: SUBSCRIPTIONS
                       name    type
            subscription_id    TEXT
                customer_id    TEXT
                    plan_id    TEXT
    subscription_start_date    TEXT
      subscription_end_date    TEXT
        subscription_status    TEXT
                 auto_renew INTEGER
          cancellation_date    TEXT
        cancellation_reason    TEXT
              monthly_price    REAL
     cancellation_date_flag    TEXT
validated_cancellation_date    TEXT
      price_validation_flag    TEXT

TABLE: SUBSCRIPTION_PLANS
          name    type
       plan_id    TEXT
     plan_name    TEXT
 monthly_price    REAL
 video_quali

In [8]:
# Business Question 1:
# What is the overall customer churn rate?

query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(churned) AS churned_customers,
    COUNT(*) - SUM(churned) AS active_customers,
    ROUND(100.0 * SUM(churned) / COUNT(*), 2) AS churn_rate_percentage
FROM churn_labels;
"""

result = pd.read_sql_query(query, conn)

display(result)

# Store the result for documentation
total = result.loc[0, "total_customers"]
churned = result.loc[0, "churned_customers"]
active = result.loc[0, "active_customers"]
rate = result.loc[0, "churn_rate_percentage"]

print("\nKEY FINDINGS")
print(f"Total customers: {total:,}")
print(f"Churned customers: {churned:,}")
print(f"Non-churned customers: {active:,}")
print(f"Overall churn rate: {rate}%")

,total_customers,churned_customers,active_customers,churn_rate_percentage
0,8000,1991,6009,24.89



KEY FINDINGS
Total customers: 8,000
Churned customers: 1,991
Non-churned customers: 6,009
Overall churn rate: 24.89%


### 4.1 Overall Customer Churn Analysis

**Business Question:** What percentage of Netflix customers have discontinued their subscriptions?

**Key Findings:**
- Total customers: 8,000
- Churned customers: 1,991
- Non-churned customers: 6,009
- Overall customer churn rate: 24.89%

**Interpretation:**

The analysis shows that approximately 25% of Netflix customers have churned. This indicates that customer retention is an important area for improvement. Understanding the reasons behind customer cancellations can help Netflix develop better retention strategies.

In [9]:
query = """
WITH ranked_subscriptions AS (
    SELECT
        customer_id,
        plan_id,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
),
latest_plan AS (
    SELECT customer_id, plan_id
    FROM ranked_subscriptions
    WHERE rn = 1
)
SELECT
    sp.plan_name,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM latest_plan lp
JOIN churn_labels cl
    ON lp.customer_id = cl.customer_id
JOIN subscription_plans sp
    ON lp.plan_id = sp.plan_id
GROUP BY sp.plan_name
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

print("\nKEY FINDINGS")
print("Churn rates are calculated using one latest subscription per customer.")
print("The plan with the highest churn rate is:",
      result.iloc[0]["plan_name"])

,plan_name,total_customers,churned_customers,churn_rate_percentage
0,Basic,3368,886,26.31
1,Standard,3018,742,24.59
2,Premium,1614,363,22.49



KEY FINDINGS
Churn rates are calculated using one latest subscription per customer.
The plan with the highest churn rate is: Basic


### 4.2 Customer Churn Analysis by Subscription Plan

**Business Question:** Which subscription plan has the highest customer churn rate?

**Key Findings:**

| Subscription Plan | Customers | Churned | Churn Rate |
|---|---:|---:|---:|
| Basic | 3,368 | 886 | 26.31% |
| Standard | 3,018 | 742 | 24.59% |
| Premium | 1,614 | 363 | 22.49% |

**Interpretation:**

The Basic plan has the highest churn rate at 26.31%, while the Premium plan has the lowest at 22.49%. This suggests that Basic plan customers may need additional engagement and retention efforts. However, the analysis shows an association, not proof that the subscription price causes churn.

**Business Recommendation:**

Netflix could investigate customer satisfaction, content preferences and payment issues among Basic plan customers to identify opportunities to improve retention.

In [10]:
query = """
SELECT
    c.country,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM customers c
JOIN churn_labels cl
    ON c.customer_id = cl.customer_id
GROUP BY c.country
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

print("\nKEY FINDINGS")
print("Country with highest churn rate:",
      result.iloc[0]["country"])
print("Highest churn rate:",
      result.iloc[0]["churn_rate_percentage"], "%")

,country,total_customers,churned_customers,churn_rate_percentage
0,South Korea,195,58,29.74
1,Indonesia,237,65,27.43
2,Germany,378,103,27.25
3,Nigeria,216,57,26.39
4,France,314,81,25.80
5,India,1251,322,25.74
6,United States,1449,370,25.53
7,Philippines,239,61,25.52
8,None,409,102,24.94
9,United Kingdom,613,151,24.63



KEY FINDINGS
Country with highest churn rate: South Korea
Highest churn rate: 29.74 %


### 4.3 Customer Churn Analysis by Country

**Business Question:** Which countries have the highest customer churn rates?

**Key Findings:**
- South Korea has the highest churn rate at 29.74%.
- Indonesia follows with 27.43%.
- Germany has a churn rate of 27.25%.
- India has 1,251 customers, of whom 322 churned (25.74%).
- The overall churn rate across all customers is 24.89%.

**Interpretation:**

South Korea, Indonesia and Germany have churn rates above the overall average. This indicates regional differences in customer retention. However, the analysis alone does not establish the reasons behind these differences.

**Business Recommendation:**

Netflix should investigate regional content preferences, pricing, payment experiences and customer satisfaction in higher-churn markets. Localized engagement and relevant content may help improve customer retention.

In [11]:
query = """
SELECT
    CASE
        WHEN c.age < 18 THEN 'Under 18'
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 35 THEN '26-35'
        WHEN c.age BETWEEN 36 AND 45 THEN '36-45'
        WHEN c.age BETWEEN 46 AND 55 THEN '46-55'
        WHEN c.age > 55 THEN 'Above 55'
        ELSE 'Unknown'
    END AS age_group,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM customers c
JOIN churn_labels cl
    ON c.customer_id = cl.customer_id
GROUP BY age_group
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,age_group,total_customers,churned_customers,churn_rate_percentage
0,46-55,774,206,26.61
1,26-35,2659,669,25.16
2,18-25,1341,332,24.76
3,Under 18,692,170,24.57
4,36-45,2063,503,24.38
5,Unknown,329,80,24.32
6,Above 55,142,31,21.83


### 4.4 Customer Churn Analysis by Age Group

**Business Question:** Which age groups have the highest customer churn rates?

**Key Findings:**

| Age Group | Customers | Churned | Churn Rate |
|---|---:|---:|---:|
| 46–55 | 774 | 206 | 26.61% |
| 26–35 | 2,659 | 669 | 25.16% |
| 18–25 | 1,341 | 332 | 24.76% |
| Under 18 | 692 | 170 | 24.57% |
| 36–45 | 2,063 | 503 | 24.38% |
| Unknown | 329 | 80 | 24.32% |
| Above 55 | 142 | 31 | 21.83% |

**Interpretation:**

Customers aged 46–55 have the highest churn rate at 26.61%, while customers above 55 have the lowest at 21.83%. The differences between most age groups are relatively small, suggesting that age alone may not be a strong indicator of customer churn.

**Business Recommendation:**

Netflix should consider customer preferences and viewing behaviour across age groups when designing engagement and retention strategies.

In [12]:
query = """
SELECT
    c.acquisition_channel,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM customers c
JOIN churn_labels cl
    ON c.customer_id = cl.customer_id
GROUP BY c.acquisition_channel
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,acquisition_channel,total_customers,churned_customers,churn_rate_percentage
0,Paid Social,1427,370,25.93
1,Direct,498,129,25.90
2,Affiliate Partner,793,200,25.22
3,Email Campaign,785,197,25.10
4,Organic Search,1786,445,24.92
5,App Store,990,240,24.24
6,Referral,1081,259,23.96
7,Partner Bundle (Telecom),640,151,23.59


### 4.5 Customer Churn Analysis by Acquisition Channel

**Business Question:** Which customer acquisition channels have the highest churn rates?

**Key Findings:**

| Acquisition Channel | Customers | Churned | Churn Rate |
|---|---:|---:|---:|
| Paid Social | 1,427 | 370 | 25.93% |
| Direct | 498 | 129 | 25.90% |
| Affiliate Partner | 793 | 200 | 25.22% |
| Email Campaign | 785 | 197 | 25.10% |
| Organic Search | 1,786 | 445 | 24.92% |
| App Store | 990 | 240 | 24.24% |
| Referral | 1,081 | 259 | 23.96% |
| Partner Bundle (Telecom) | 640 | 151 | 23.59% |

**Interpretation:**

Paid Social has the highest churn rate at 25.93%, closely followed by Direct at 25.90%. Partner Bundle (Telecom) has the lowest churn rate at 23.59%. The relatively small differences suggest that acquisition channel alone does not explain customer churn.

**Business Recommendation:**

Netflix should evaluate customer retention across acquisition channels and examine whether customers acquired through paid campaigns receive enough engagement and value after joining.

In [13]:
query = """
WITH latest_subscriptions AS (
    SELECT
        customer_id,
        auto_renew,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
)
SELECT
    CASE
        WHEN ls.auto_renew = 1 THEN 'Enabled'
        WHEN ls.auto_renew = 0 THEN 'Disabled'
        ELSE 'Unknown'
    END AS auto_renew_status,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM latest_subscriptions ls
JOIN churn_labels cl
    ON ls.customer_id = cl.customer_id
WHERE ls.rn = 1
GROUP BY auto_renew_status
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,auto_renew_status,total_customers,churned_customers,churn_rate_percentage
0,Disabled,1470,422,28.71
1,Enabled,6530,1569,24.03


### 4.6 Customer Churn Analysis by Auto-Renewal Status

**Business Question:** Do customers with auto-renewal enabled have a different churn rate from customers without auto-renewal?

**Key Findings:**

| Auto-Renewal Status | Customers | Churned | Churn Rate |
|---|---:|---:|---:|
| Disabled | 1,470 | 422 | 28.71% |
| Enabled | 6,530 | 1,569 | 24.03% |

**Interpretation:**

Customers with auto-renewal disabled have a churn rate of 28.71%, compared with 24.03% for customers with auto-renewal enabled. This is a difference of 4.68 percentage points. Auto-renewal may be associated with better customer retention, although this analysis does not establish causation.

**Business Recommendation:**

Netflix could encourage customers to enable auto-renewal through clear reminders and convenient payment options while ensuring customers retain control over their subscriptions.

In [14]:
query = """
SELECT
    churn_reason,
    COUNT(*) AS churned_customers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM churn_labels WHERE churned = 1),
        2
    ) AS percentage_of_churn
FROM churn_labels
WHERE churned = 1
GROUP BY churn_reason
ORDER BY churned_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,churn_reason,churned_customers,percentage_of_churn
0,Too expensive,365,18.33
1,Found alternative platform,284,14.26
2,Not enough content,217,10.90
3,No longer needed,212,10.65
4,Technical issues,182,9.14
5,Household budget cuts,145,7.28
6,Payment issues,139,6.98
7,Content quality decline,139,6.98
8,Poor customer service experience,127,6.38
9,Moving / relocation,98,4.92


### 4.7 Analysis of Customer Churn Reasons

**Business Question:** What are the most common reasons customers discontinue their Netflix subscriptions?

**Key Findings:**

| Churn Reason | Customers | Percentage |
|---|---:|---:|
| Too expensive | 365 | 18.33% |
| Found alternative platform | 284 | 14.26% |
| Not enough content | 217 | 10.90% |
| No longer needed | 212 | 10.65% |
| Technical issues | 182 | 9.14% |
| Household budget cuts | 145 | 7.28% |
| Payment issues | 139 | 6.98% |
| Content quality decline | 139 | 6.98% |
| Poor customer service experience | 127 | 6.38% |
| Moving / relocation | 98 | 4.92% |
| Password sharing crackdown | 83 | 4.17% |

**Interpretation:**

Price is the leading recorded reason for customer churn, accounting for 18.33% of cancellations. Competition from alternative platforms is the second-largest reason at 14.26%. Content availability, technical issues and customer service also contribute to cancellations.

**Business Recommendations:**
- Review pricing and subscription value.
- Improve content variety and quality.
- Investigate technical problems affecting viewing experiences.
- Improve customer support and payment reliability.
- Monitor competitor offerings and customer expectations.

In [15]:
query = """
WITH payment_summary AS (
    SELECT
        customer_id,
        SUM(CASE WHEN payment_status = 'Failed' THEN 1 ELSE 0 END)
            AS failed_payments
    FROM payments
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN COALESCE(ps.failed_payments, 0) > 0
            THEN 'Has Failed Payments'
        ELSE 'No Failed Payments'
    END AS payment_failure_group,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM churn_labels cl
LEFT JOIN payment_summary ps
    ON cl.customer_id = ps.customer_id
GROUP BY payment_failure_group
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,payment_failure_group,total_customers,churned_customers,churn_rate_percentage
0,Has Failed Payments,4533,1287,28.39
1,No Failed Payments,3467,704,20.31


### 4.8 Analysis of Payment Failures and Customer Churn

**Business Question:** Do customers who experience failed payments have a higher churn rate?

**Key Findings:**

| Payment Failure Group | Total Customers | Churned Customers | Churn Rate |
|---|---:|---:|---:|
| Has Failed Payments | 4,533 | 1,287 | 28.39% |
| No Failed Payments | 3,467 | 704 | 20.31% |

**Interpretation:**

Customers with failed payment records have a churn rate of 28.39%, compared with 20.31% among customers without failed payments. This is a difference of 8.08 percentage points. Payment failures are associated with higher customer churn in this dataset, although this analysis alone does not prove that failed payments cause cancellations.

**Business Recommendations:**
- Improve payment processing reliability.
- Notify customers when a payment fails.
- Provide simple payment retry options.
- Offer alternative payment methods.
- Monitor customers with repeated payment failures and provide timely support.

In [16]:
query = """
SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN payment_status = 'Failed' THEN 1 ELSE 0 END)
        AS failed_transactions,
    ROUND(
        100.0 * SUM(CASE WHEN payment_status = 'Failed' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS failure_rate_percentage
FROM payments
GROUP BY payment_method
ORDER BY total_transactions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,payment_method,total_transactions,failed_transactions,failure_rate_percentage
0,Credit Card,33249,2145,6.45
1,Debit Card,19751,1282,6.49
2,UPI,17730,1137,6.41
3,PayPal,13375,853,6.38
4,Net Banking,11344,742,6.54
5,Mobile Wallet,8697,570,6.55
6,Gift Card,6589,477,7.24


### 4.9 Analysis of Payment Methods and Transaction Failures

**Business Question:** Which payment methods are most commonly used, and what are their payment failure rates?

**Key Findings:**

| Payment Method | Total Transactions | Failed Transactions | Failure Rate |
|---|---:|---:|---:|
| Credit Card | 33,249 | 2,145 | 6.45% |
| Debit Card | 19,751 | 1,282 | 6.49% |
| UPI | 17,730 | 1,137 | 6.41% |
| PayPal | 13,375 | 853 | 6.38% |
| Net Banking | 11,344 | 742 | 6.54% |
| Mobile Wallet | 8,697 | 570 | 6.55% |
| Gift Card | 6,589 | 477 | 7.24% |

**Interpretation:**

Credit cards are the most frequently used payment method, with 33,249 transactions. Most payment methods have similar failure rates, ranging from 6.38% to 6.55%. Gift cards have the highest failure rate at 7.24%.

**Business Recommendations:**
- Investigate why gift card transactions have a higher failure rate.
- Monitor payment failures across all payment methods.
- Improve transaction reliability and error handling.
- Continue supporting multiple payment options for customer convenience.

In [19]:
query = """
SELECT
    strftime('%Y-%m', payment_date) AS payment_month,
    COUNT(*) AS total_transactions,
    SUM(CASE
        WHEN payment_status = 'Success' THEN 1
        ELSE 0
    END) AS successful_transactions,
    ROUND(
        SUM(CASE
            WHEN payment_status = 'Success' THEN amount
            ELSE 0
        END), 2
    ) AS successful_payment_amount
FROM payments
GROUP BY strftime('%Y-%m', payment_date)
ORDER BY payment_month;
"""

result = pd.read_sql_query(query, conn)
display(result)

,payment_month,total_transactions,successful_transactions,successful_payment_amount
0,2019-01,96,88,1164.13
1,2019-02,159,145,1913.06
2,2019-03,269,258,3645.22
3,2019-04,330,305,3950.73
4,2019-05,445,421,5580.56
...,...,...,...,...
84,2026-01,1355,1264,17712.42
85,2026-02,1204,1138,16174.38
86,2026-03,1379,1297,18336.66
87,2026-04,1324,1228,17230.67


### 4.10 Monthly Payment Revenue Analysis

**Business Question:** How does Netflix's recorded successful payment revenue change from month to month?

**Key Findings:**
- The analysis covers 89 months, from January 2019 to May 2026.
- The dataset contains 103,529 successful transactions.
- The total amount from successful transactions across the dataset is 1,430,511.33.
- In January 2019, successful payment revenue was 1,164.13 from 88 transactions.
- In May 2026, successful payment revenue was 18,213.53 from 1,280 transactions.

**Interpretation:**

The monthly results show a substantial increase in recorded successful payment activity between January 2019 and May 2026. The number of successful transactions and the corresponding payment amount are both higher in the latest displayed month. This indicates growth in recorded payment activity, although payment revenue alone does not establish the number of active subscribers or overall profitability.

**Business Recommendations:**
- Monitor monthly successful payment revenue and transaction volume.
- Investigate months with unusual increases or decreases.
- Track failed transactions alongside successful payments.
- Use monthly payment trends to support subscription and revenue planning.

**Note:** The payment amounts represent the values recorded in the dataset and should not automatically be interpreted as audited company revenue.

In [20]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_viewing_sessions,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(AVG(watch_duration_minutes), 2) AS average_watch_minutes,
    ROUND(SUM(watch_duration_minutes), 2) AS total_watch_minutes
FROM viewing_activity
GROUP BY device_type
ORDER BY total_viewing_sessions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,device_type,total_viewing_sessions,unique_customers,average_watch_minutes,total_watch_minutes
0,Smart TV,25132,7419,49.36,1240454.0
1,Mobile,19822,7059,49.02,971765.0
2,Laptop,8333,5053,50.38,419847.0
3,Tablet,7811,4862,49.77,388783.0
4,Gaming Console,3959,3054,49.02,194079.0
5,Desktop,701,672,51.52,36118.0
6,None,664,635,48.87,32450.0


### 4.11 Customer Viewing Engagement by Device Type

**Business Question:** Which devices do customers use most frequently to watch Netflix, and how much time do they spend watching?

**Key Findings:**

| Device Type | Viewing Sessions | Unique Customers | Average Watch Time (Minutes) |
|---|---:|---:|---:|
| Smart TV | 25,132 | 7,419 | 49.36 |
| Mobile | 19,822 | 7,059 | 49.02 |
| Laptop | 8,333 | 5,053 | 50.38 |
| Tablet | 7,811 | 4,862 | 49.77 |
| Gaming Console | 3,959 | 3,054 | 49.02 |
| Desktop | 701 | 672 | 51.52 |
| Unknown | 664 | 635 | 48.87 |

**Interpretation:**

Smart TVs account for the highest number of viewing sessions, followed by mobile devices. Customers use multiple device types to access Netflix. Average viewing duration is relatively similar across devices, ranging from approximately 49 to 52 minutes per session.

**Business Recommendations:**
- Continue improving the Smart TV and mobile viewing experience.
- Ensure consistent streaming quality across devices.
- Investigate missing device information to improve data quality.
- Use device preferences when planning customer engagement strategies.

In [21]:
query = """
WITH customer_viewing AS (
    SELECT
        customer_id,
        SUM(watch_duration_minutes) AS total_watch_minutes,
        AVG(watch_duration_minutes) AS average_watch_minutes,
        COUNT(*) AS total_viewing_sessions
    FROM viewing_activity
    GROUP BY customer_id
),
customer_groups AS (
    SELECT
        customer_id,
        total_watch_minutes,
        CASE
            WHEN total_watch_minutes < 500 THEN 'Low Engagement'
            WHEN total_watch_minutes < 1500 THEN 'Medium Engagement'
            ELSE 'High Engagement'
        END AS engagement_group
    FROM customer_viewing
)
SELECT
    cg.engagement_group,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage,
    ROUND(AVG(cg.total_watch_minutes), 2) AS average_watch_minutes
FROM customer_groups cg
JOIN churn_labels cl
    ON cg.customer_id = cl.customer_id
GROUP BY cg.engagement_group
ORDER BY average_watch_minutes;
"""

result = pd.read_sql_query(query, conn)
display(result)

,engagement_group,total_customers,churned_customers,churn_rate_percentage,average_watch_minutes
0,Low Engagement,5645,1474,26.11,312.20
1,Medium Engagement,2327,507,21.79,647.72
2,High Engagement,8,1,12.50,1733.88


### 4.12 Relationship Between Viewing Engagement and Customer Churn

**Business Question:** Do customers who spend more time watching Netflix have a lower churn rate?

**Key Findings:**

| Engagement Group | Total Customers | Churned Customers | Churn Rate | Average Watch Minutes |
|---|---:|---:|---:|---:|
| Low Engagement | 5,645 | 1,474 | 26.11% | 312.20 |
| Medium Engagement | 2,327 | 507 | 21.79% | 647.72 |
| High Engagement | 8 | 1 | 12.50% | 1,733.88 |

**Interpretation:**

The results show that customers in the low-engagement group have the highest churn rate at 26.11%. The medium-engagement group has a lower churn rate of 21.79%. The high-engagement group has the lowest rate at 12.50%, but it contains only eight customers, so this result is not sufficiently representative for a strong conclusion.

Overall, the results suggest that lower viewing engagement is associated with higher customer churn.

**Business Recommendations:**
- Identify customers whose viewing activity is declining.
- Recommend content based on customer interests.
- Use personalized notifications to encourage engagement.
- Monitor inactive customers and offer relevant content suggestions.
- Collect more data before drawing conclusions about highly engaged customers.

In [22]:
query = """
SELECT
    issue_category,
    COUNT(*) AS total_tickets,
    ROUND(AVG(resolution_time_hours), 2) AS average_resolution_hours,
    ROUND(AVG(customer_satisfaction_score), 2) AS average_satisfaction_score
FROM support_tickets
GROUP BY issue_category
ORDER BY total_tickets DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,issue_category,total_tickets,average_resolution_hours,average_satisfaction_score
0,Billing & Payments,1709,17.30,2.98
1,Streaming Quality,1323,18.57,3.04
2,Account Access,998,17.78,3.47
3,Subscription,881,18.24,3.50
4,Device & App,738,18.40,3.56
5,Content,471,18.35,3.55


### 4.13 Customer Support Ticket Analysis

**Business Question:** Which customer support issues are reported most frequently, and how quickly are they resolved?

**Key Findings:**

| Issue Category | Total Tickets | Average Resolution Time (Hours) | Average Satisfaction Score |
|---|---:|---:|---:|
| Billing & Payments | 1,709 | 17.30 | 2.98 |
| Streaming Quality | 1,323 | 18.57 | 3.04 |
| Account Access | 998 | 17.78 | 3.47 |
| Subscription | 881 | 18.24 | 3.50 |
| Device & App | 738 | 18.40 | 3.56 |
| Content | 471 | 18.35 | 3.55 |

**Interpretation:**

Billing and Payments is the most frequently reported support issue, with 1,709 tickets. Streaming Quality is the second most common issue. Both categories also have relatively low customer satisfaction scores of 2.98 and 3.04, respectively.

Device and App issues have the highest average satisfaction score at 3.56. Average resolution times are relatively similar across all categories, ranging from 17.30 to 18.57 hours.

**Business Recommendations:**
- Investigate recurring billing and payment problems.
- Improve streaming quality and troubleshoot common technical issues.
- Review support processes for low-satisfaction categories.
- Monitor resolution times and customer satisfaction together.
- Develop preventive solutions for frequently reported problems.

In [23]:
query = """
WITH customer_support AS (
    SELECT
        customer_id,
        AVG(customer_satisfaction_score) AS average_satisfaction,
        COUNT(*) AS total_support_tickets
    FROM support_tickets
    WHERE customer_satisfaction_score IS NOT NULL
    GROUP BY customer_id
),
satisfaction_groups AS (
    SELECT
        customer_id,
        average_satisfaction,
        CASE
            WHEN average_satisfaction < 3 THEN 'Low Satisfaction'
            WHEN average_satisfaction < 4 THEN 'Medium Satisfaction'
            ELSE 'High Satisfaction'
        END AS satisfaction_group
    FROM customer_support
)
SELECT
    sg.satisfaction_group,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage,
    ROUND(AVG(sg.average_satisfaction), 2) AS average_satisfaction_score
FROM satisfaction_groups sg
JOIN churn_labels cl
    ON sg.customer_id = cl.customer_id
GROUP BY sg.satisfaction_group
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,satisfaction_group,total_customers,churned_customers,churn_rate_percentage,average_satisfaction_score
0,Low Satisfaction,896,301,33.59,1.86
1,Medium Satisfaction,1283,348,27.12,3.09
2,High Satisfaction,1361,230,16.90,4.32


### 4.14 Relationship Between Customer Support Satisfaction and Churn

**Business Question:** Are customers who give lower support satisfaction scores more likely to churn?

**Key Findings:**

| Satisfaction Group | Total Customers | Churned Customers | Churn Rate | Average Satisfaction Score |
|---|---:|---:|---:|---:|
| Low Satisfaction | 896 | 301 | 33.59% | 1.86 |
| Medium Satisfaction | 1,283 | 348 | 27.12% | 3.09 |
| High Satisfaction | 1,361 | 230 | 16.90% | 4.32 |

**Interpretation:**

Customers with low support satisfaction have the highest churn rate at 33.59%. Customers with high satisfaction have the lowest churn rate at 16.90%. This shows a clear association between lower support satisfaction and higher customer churn in the analyzed data.

**Business Recommendations:**
- Improve the quality of customer support.
- Prioritize customers who report low satisfaction.
- Resolve recurring customer complaints more effectively.
- Collect feedback after resolving support tickets.
- Monitor satisfaction scores to identify customers who may be at risk of leaving.

In [24]:
query = """
SELECT
    sentiment_label,
    COUNT(*) AS total_feedback,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_feedback),
        2
    ) AS feedback_percentage,
    ROUND(AVG(rating), 2) AS average_rating
FROM customer_feedback
GROUP BY sentiment_label
ORDER BY total_feedback DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,sentiment_label,total_feedback,feedback_percentage,average_rating
0,Positive,1978,39.20,4.40
1,Neutral,1606,31.83,3.03
2,Negative,1462,28.97,1.64


### 4.15 Customer Feedback Sentiment Analysis

**Business Question:** What is the distribution of positive, neutral, and negative customer feedback?

**Key Findings:**

| Sentiment | Total Feedback | Percentage | Average Rating |
|---|---:|---:|---:|
| Positive | 1,978 | 39.20% | 4.40 |
| Neutral | 1,606 | 31.83% | 3.03 |
| Negative | 1,462 | 28.97% | 1.64 |

**Interpretation:**

Positive feedback represents the largest group, accounting for 39.20% of all feedback. However, 28.97% of feedback is negative, with an average rating of 1.64. This indicates that although many customers are satisfied, a considerable proportion have expressed dissatisfaction.

**Business Recommendations:**
- Review negative feedback to identify recurring complaints.
- Improve services and features associated with low ratings.
- Continue monitoring customer sentiment over time.
- Use customer feedback to guide improvements in content, streaming quality and support.

In [25]:
query = """
WITH customer_sentiment AS (
    SELECT
        customer_id,
        AVG(rating) AS average_rating,
        COUNT(*) AS feedback_count,
        CASE
            WHEN AVG(rating) < 2.5 THEN 'Negative Feedback'
            WHEN AVG(rating) < 3.5 THEN 'Neutral Feedback'
            ELSE 'Positive Feedback'
        END AS feedback_group
    FROM customer_feedback
    GROUP BY customer_id
)
SELECT
    cs.feedback_group,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage,
    ROUND(AVG(cs.average_rating), 2) AS average_rating
FROM customer_sentiment cs
JOIN churn_labels cl
    ON cs.customer_id = cl.customer_id
GROUP BY cs.feedback_group
ORDER BY churn_rate_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,feedback_group,total_customers,churned_customers,churn_rate_percentage,average_rating
0,Negative Feedback,917,278,30.32,1.58
1,Neutral Feedback,1170,309,26.41,2.96
2,Positive Feedback,1474,305,20.69,4.34


### 4.16 Relationship Between Customer Feedback and Churn

**Business Question:** Are customers who provide negative feedback more likely to churn?

**Key Findings:**

| Feedback Group | Total Customers | Churned Customers | Churn Rate | Average Rating |
|---|---:|---:|---:|---:|
| Negative Feedback | 917 | 278 | 30.32% | 1.58 |
| Neutral Feedback | 1,170 | 309 | 26.41% | 2.96 |
| Positive Feedback | 1,474 | 305 | 20.69% | 4.34 |

**Interpretation:**

Customers who provide negative feedback have the highest churn rate at 30.32%, while customers who provide positive feedback have the lowest rate at 20.69%. This suggests that customer dissatisfaction is associated with a greater likelihood of subscription cancellation.

**Business Recommendations:**
- Identify customers who repeatedly provide negative feedback.
- Investigate common complaints and address their root causes.
- Improve customer experience based on feedback patterns.
- Use feedback ratings to identify customers who may be at risk of churning.
- Monitor whether improvements lead to better satisfaction and retention.

In [26]:
query = """
SELECT
    c.genre,
    COUNT(v.viewing_id) AS total_viewing_sessions,
    COUNT(DISTINCT v.customer_id) AS unique_viewers,
    ROUND(AVG(v.watch_duration_minutes), 2) AS average_watch_minutes,
    ROUND(AVG(v.completion_percentage), 2) AS average_completion_percentage
FROM viewing_activity v
JOIN content c
    ON v.content_id = c.content_id
GROUP BY c.genre
ORDER BY total_viewing_sessions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,genre,total_viewing_sessions,unique_viewers,average_watch_minutes,average_completion_percentage
0,Anime,5556,3935,49.07,70.38
1,Horror,5325,3782,48.59,70.51
2,Comedy,5202,3758,49.90,70.77
3,Family,4820,3573,49.39,70.52
4,Sci-Fi,4446,3355,49.85,70.48
5,Thriller,4381,3326,49.98,70.32
6,Stand-Up Comedy,4273,3259,49.58,70.20
7,Documentary,4224,3206,48.62,70.28
8,Reality,3896,3052,49.71,69.99
9,Fantasy,3886,3029,48.57,70.89


### 4.17 Content Genre Popularity and Viewing Engagement

**Business Question:** Which content genres receive the highest viewing activity?

**Key Findings:**

| Genre | Viewing Sessions | Unique Viewers | Average Watch Minutes |
|---|---:|---:|---:|
| Anime | 5,556 | 3,935 | 49.07 |
| Horror | 5,325 | 3,782 | 48.59 |
| Comedy | 5,202 | 3,758 | 49.90 |
| Family | 4,820 | 3,573 | 49.39 |
| Sci-Fi | 4,446 | 3,355 | 49.85 |
| Thriller | 4,381 | 3,326 | 49.98 |
| Stand-Up Comedy | 4,273 | 3,259 | 49.58 |
| Documentary | 4,224 | 3,206 | 48.62 |
| Reality | 3,896 | 3,052 | 49.71 |
| Fantasy | 3,886 | 3,029 | 48.57 |
| Sports | 3,849 | 3,011 | 49.40 |
| Mystery | 3,829 | 3,023 | 49.18 |
| Action | 3,565 | 2,858 | 49.97 |
| Romance | 3,327 | 2,681 | 48.96 |
| Crime | 2,990 | 2,464 | 50.81 |
| Drama | 2,853 | 2,374 | 50.06 |

**Interpretation:**

Anime has the highest viewing activity, with 5,556 sessions and 3,935 unique viewers. Horror and Comedy are the next most-watched genres. Crime has the highest average watch duration at 50.81 minutes, while average completion percentages are relatively similar across genres.

**Business Recommendations:**
- Analyze popular genres when planning content acquisition and recommendations.
- Promote Anime, Horror and Comedy to interested audiences.
- Explore why Crime and Drama have relatively high average watch durations.
- Use individual viewing preferences to personalize content recommendations.
- Continue monitoring genre popularity and completion rates.

In [27]:
query = """
SELECT
    c.content_type,
    COUNT(v.viewing_id) AS total_viewing_sessions,
    COUNT(DISTINCT v.customer_id) AS unique_viewers,
    COUNT(DISTINCT c.content_id) AS unique_content_titles,
    ROUND(AVG(v.watch_duration_minutes), 2) AS average_watch_minutes,
    ROUND(AVG(v.completion_percentage), 2) AS average_completion_percentage
FROM viewing_activity v
JOIN content c
    ON v.content_id = c.content_id
GROUP BY c.content_type
ORDER BY total_viewing_sessions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,content_type,total_viewing_sessions,unique_viewers,unique_content_titles,average_watch_minutes,average_completion_percentage
0,Movie,28754,7543,216,49.44,70.42
1,TV Show,19012,7010,144,49.71,70.60
2,Limited Series,7888,4913,59,49.29,70.57
3,Documentary,7436,4765,55,48.92,70.55
4,Stand-Up Special,3332,2684,26,49.27,69.83


### 4.18 Content Type Analysis

**Business Question:** Do customers watch movies or TV shows more frequently?

**Key Findings:**

| Content Type | Viewing Sessions | Unique Viewers | Content Titles | Average Watch Minutes |
|---|---:|---:|---:|---:|
| Movie | 28,754 | 7,543 | 216 | 49.44 |
| TV Show | 19,012 | 7,010 | 144 | 49.71 |
| Limited Series | 7,888 | 4,913 | 59 | 49.29 |
| Documentary | 7,436 | 4,765 | 55 | 48.92 |
| Stand-Up Special | 3,332 | 2,684 | 26 | 49.27 |

**Interpretation:**

Movies account for the highest viewing activity, with 28,754 sessions and 7,543 unique viewers. TV Shows are the second most-watched content type. Average viewing duration and completion percentages are very similar across all content types.

**Business Recommendations:**
- Continue offering a variety of movies and TV shows.
- Use viewing preferences to recommend relevant content.
- Monitor the popularity of limited series and documentaries.
- Evaluate content performance using viewing sessions and completion rates.

In [28]:
query = """
SELECT
    c.content_language,
    COUNT(v.viewing_id) AS total_viewing_sessions,
    COUNT(DISTINCT v.customer_id) AS unique_viewers,
    ROUND(AVG(v.watch_duration_minutes), 2) AS average_watch_minutes,
    ROUND(AVG(v.completion_percentage), 2) AS average_completion_percentage
FROM viewing_activity v
JOIN content c
    ON v.content_id = c.content_id
GROUP BY c.content_language
ORDER BY total_viewing_sessions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,content_language,total_viewing_sessions,unique_viewers,average_watch_minutes,average_completion_percentage
0,English,22712,7323,49.37,70.61
1,Hindi,11551,5923,49.84,70.37
2,Spanish,5650,3919,49.46,70.37
3,Portuguese,4657,3469,49.78,70.53
4,Korean,4325,3301,49.13,70.44
5,French,3909,3060,49.27,70.18
6,German,3590,2887,48.02,71.07
7,Arabic,2847,2366,49.41,70.78
8,Indonesian,2471,2120,50.60,69.66
9,Japanese,1897,1670,49.67,69.24


### 4.19 Content Language Analysis

**Business Question:** Which content languages attract the highest number of viewers?

**Key Findings:**

| Content Language | Viewing Sessions | Unique Viewers | Average Watch Minutes |
|---|---:|---:|---:|
| English | 22,712 | 7,323 | 49.37 |
| Hindi | 11,551 | 5,923 | 49.84 |
| Spanish | 5,650 | 3,919 | 49.46 |
| Portuguese | 4,657 | 3,469 | 49.78 |
| Korean | 4,325 | 3,301 | 49.13 |
| French | 3,909 | 3,060 | 49.27 |
| German | 3,590 | 2,887 | 48.02 |
| Arabic | 2,847 | 2,366 | 49.41 |
| Indonesian | 2,471 | 2,120 | 50.60 |
| Japanese | 1,897 | 1,670 | 49.67 |
| Tagalog | 1,605 | 1,440 | 49.33 |
| Italian | 679 | 648 | 47.34 |
| Polish | 529 | 513 | 50.31 |

**Interpretation:**

English-language content has the highest viewing activity, with 22,712 sessions and 7,323 unique viewers. Hindi is the second most-watched language. Several other languages, including Spanish, Portuguese and Korean, also attract substantial viewing activity.

Average viewing duration is relatively similar across languages, suggesting that customers engage with content in different languages for comparable periods.

**Business Recommendations:**
- Continue offering a strong selection of English and Hindi content.
- Expand relevant regional and international content.
- Use customer language preferences for personalized recommendations.
- Monitor viewing activity across languages to guide content planning.

In [29]:
query = """
SELECT
    ticket_status,
    COUNT(*) AS total_tickets,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM support_tickets),
        2
    ) AS ticket_percentage,
    ROUND(AVG(resolution_time_hours), 2) AS average_resolution_hours
FROM support_tickets
GROUP BY ticket_status
ORDER BY total_tickets DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,ticket_status,total_tickets,ticket_percentage,average_resolution_hours
0,Resolved,3390,55.39,18.05
1,Closed,1202,19.64,17.85
2,Pending,888,14.51,NaN
3,Escalated,640,10.46,NaN


### 4.20 Customer Support Ticket Status Analysis

**Business Question:** What is the current status of customer support tickets?

**Key Findings:**

| Ticket Status | Total Tickets | Percentage | Average Resolution Time (Hours) |
|---|---:|---:|---:|
| Resolved | 3,390 | 55.39% | 18.05 |
| Closed | 1,202 | 19.64% | 17.85 |
| Pending | 888 | 14.51% | Not available |
| Escalated | 640 | 10.46% | Not available |

**Interpretation:**

Resolved tickets represent 55.39% of all support tickets, while 19.64% are marked as closed. However, 14.51% remain pending and 10.46% have been escalated. Resolution times are unavailable for pending and escalated tickets, which may indicate that these issues have not yet been resolved or that resolution information has not been recorded.

**Business Recommendations:**
- Prioritize pending customer support tickets.
- Review escalated cases to identify complex or recurring issues.
- Monitor unresolved tickets to prevent long waiting times.
- Improve tracking of ticket status and resolution information.
- Review support team performance regularly.

In [30]:
query = """
WITH latest_subscription AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
)
SELECT
    subscription_status,
    COUNT(*) AS total_customers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM latest_subscription WHERE rn = 1),
        2
    ) AS customer_percentage
FROM latest_subscription
WHERE rn = 1
GROUP BY subscription_status
ORDER BY total_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,subscription_status,total_customers,customer_percentage
0,Active,6009,75.11
1,Cancelled,1991,24.89


### 4.21 Customer Distribution by Subscription Status

**Business Question:** What is the distribution of customers across subscription statuses?

**Key Findings:**

| Subscription Status | Total Customers | Percentage |
|---|---:|---:|
| Active | 6,009 | 75.11% |
| Cancelled | 1,991 | 24.89% |

**Interpretation:**

Out of 8,000 customers, 6,009 (75.11%) have active subscriptions, while 1,991 (24.89%) have cancelled their subscriptions. This matches the overall churn rate identified earlier in the analysis.

**Business Recommendations:**
- Focus on retaining active customers.
- Identify customers who may be at risk of cancellation.
- Analyze cancellation reasons to improve retention strategies.
- Monitor subscription status changes regularly.

In [31]:
query = """
WITH latest_subscription AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
)
SELECT
    sp.plan_name,
    COUNT(*) AS total_customers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM latest_subscription WHERE rn = 1),
        2
    ) AS customer_percentage,
    ROUND(AVG(ls.monthly_price), 2) AS average_monthly_price
FROM latest_subscription ls
JOIN subscription_plans sp
    ON ls.plan_id = sp.plan_id
WHERE ls.rn = 1
GROUP BY sp.plan_name
ORDER BY total_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,plan_name,total_customers,customer_percentage,average_monthly_price
0,Basic,3368,42.10,6.99
1,Standard,3018,37.73,15.49
2,Premium,1614,20.18,22.99


### 4.22 Subscription Plan Popularity Analysis

**Business Question:** Which subscription plans are most popular among customers?

**Key Findings:**

| Subscription Plan | Customers | Percentage | Average Monthly Price |
|---|---:|---:|---:|
| Basic | 3,368 | 42.10% | 6.99 |
| Standard | 3,018 | 37.73% | 15.49 |
| Premium | 1,614 | 20.18% | 22.99 |

**Interpretation:**

The Basic plan is the most popular, accounting for 42.10% of customers. The Standard plan follows at 37.73%, while Premium has the smallest share at 20.18%. This indicates that more customers choose the lower-priced plans.

**Business Recommendations:**
- Understand what Basic plan customers value most.
- Consider relevant upgrades from Basic to Standard.
- Evaluate Premium plan benefits and customer demand.
- Maintain a balance between affordable pricing and plan features.

In [32]:
query = """
WITH subscription_durations AS (
    SELECT
        customer_id,
        subscription_id,
        julianday(
            COALESCE(
                validated_cancellation_date,
                subscription_end_date,
                DATE('now')
            )
        ) - julianday(subscription_start_date) AS duration_days
    FROM subscriptions
)
SELECT
    COUNT(*) AS total_subscriptions,
    ROUND(AVG(duration_days), 2) AS average_duration_days,
    ROUND(AVG(duration_days) / 30.44, 2) AS average_duration_months,
    ROUND(MIN(duration_days), 2) AS minimum_duration_days,
    ROUND(MAX(duration_days), 2) AS maximum_duration_days
FROM subscription_durations
WHERE duration_days >= 0;
"""

result = pd.read_sql_query(query, conn)
display(result)

,total_subscriptions,average_duration_days,average_duration_months,minimum_duration_days,maximum_duration_days
0,8875,1295.82,42.57,0.0,2860.0


### 4.23 Customer Subscription Duration Analysis

**Business Question:** What is the average duration of customer subscriptions?

**Key Findings:**

| Metric | Result |
|---|---:|
| Total Subscriptions | 8,875 |
| Average Duration | 1,295.82 days |
| Average Duration in Months | 42.57 months |
| Minimum Duration | 0 days |
| Maximum Duration | 2,860 days |

**Interpretation:**

The average recorded subscription duration is approximately 42.57 months (3.55 years). The duration ranges from 0 to 2,860 days. This suggests that subscription records include both newly started and long-running subscriptions.

**Business Recommendations:**
- Study the behaviour of long-term subscribers to understand retention factors.
- Identify customers who cancel shortly after subscribing.
- Improve the initial customer experience to encourage longer subscriptions.
- Review subscription duration alongside churn reasons and customer engagement.

**Note:** The calculation uses the available cancellation date, subscription end date, or the current date for active records. It represents recorded subscription duration, not necessarily uninterrupted customer tenure.

In [33]:
query = """
SELECT
    strftime('%Y-%m', registration_date) AS registration_month,
    COUNT(*) AS new_customers
FROM customers
WHERE registration_date IS NOT NULL
GROUP BY strftime('%Y-%m', registration_date)
ORDER BY registration_month;
"""

result = pd.read_sql_query(query, conn)
display(result)

,registration_month,new_customers
0,2019-01,91
1,2019-02,82
2,2019-03,86
3,2019-04,71
4,2019-05,98
...,...,...
84,2026-01,84
85,2026-02,101
86,2026-03,103
87,2026-04,79


### 4.24 Monthly Customer Registration Analysis

**Business Question:** How many new customers registered each month?

**Key Findings:**
- The analysis covers 89 months, from January 2019 to May 2026.
- Monthly registrations vary over the period.
- Recent monthly registrations include:
  - January 2026: 84 customers
  - February 2026: 101 customers
  - March 2026: 103 customers
  - April 2026: 79 customers
  - May 2026: 83 customers

**Interpretation:**

Customer registrations fluctuate from month to month. March 2026 recorded 103 new customers among the recent months shown. The complete monthly trend can help identify periods of stronger or weaker customer acquisition.

**Business Recommendations:**
- Monitor monthly registration trends to evaluate customer acquisition.
- Investigate months with unusually high or low registrations.
- Compare registration trends with marketing campaigns and acquisition channels.
- Use registration trends to support customer growth planning.

**Note:** The displayed output is abbreviated by Pandas. The findings above describe the visible recent months and the full analysis period; they do not claim an overall peak or minimum.

In [34]:
query = """
SELECT
    strftime('%Y-%m', churn_date) AS churn_month,
    COUNT(*) AS churned_customers
FROM churn_labels
WHERE churned = 1
  AND churn_date IS NOT NULL
GROUP BY strftime('%Y-%m', churn_date)
ORDER BY churn_month;
"""

result = pd.read_sql_query(query, conn)
display(result)

,churn_month,churned_customers
0,2025-10,198
1,2025-11,200
2,2025-12,213
3,2026-01,253
4,2026-02,190
5,2026-03,213
6,2026-04,223
7,2026-05,250
8,2026-06,251


### 4.25 Monthly Customer Churn Trend Analysis

**Business Question:** How many customers churned each month?

**Key Findings:**

The churn data contains monthly records from October 2025 to June 2026.

| Month | Churned Customers |
|---|---:|
| October 2025 | 198 |
| November 2025 | 200 |
| December 2025 | 213 |
| January 2026 | 253 |
| February 2026 | 190 |
| March 2026 | 213 |
| April 2026 | 223 |
| May 2026 | 250 |
| June 2026 | 251 |

**Interpretation:**

Monthly churn varied during the observed period. January 2026 recorded the highest number of cancellations (253), while February recorded the lowest (190). Churn increased again between February and June 2026.

**Business Recommendations:**
- Monitor monthly churn to identify increases early.
- Investigate the reasons behind higher-churn months.
- Review customer feedback, payment failures, and support issues.
- Introduce targeted retention campaigns during periods of rising churn.

**Note:** This analysis covers only the months available in the churn-date data. It does not establish the cause of monthly changes.

In [35]:
query = """
WITH latest_subscription AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
)
SELECT
    sp.plan_name,
    ls.cancellation_reason,
    COUNT(*) AS cancelled_customers
FROM latest_subscription ls
JOIN subscription_plans sp
    ON ls.plan_id = sp.plan_id
WHERE ls.rn = 1
  AND ls.subscription_status = 'Cancelled'
  AND ls.cancellation_reason IS NOT NULL
GROUP BY sp.plan_name, ls.cancellation_reason
ORDER BY sp.plan_name, cancelled_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,plan_name,cancellation_reason,cancelled_customers
0,Basic,Too expensive,201
1,Basic,Found alternative platform,111
2,Basic,Not enough content,105
3,Basic,No longer needed,85
4,Basic,Technical issues,69
5,Basic,Payment issues,67
6,Basic,Content quality decline,63
7,Basic,Poor customer service experience,56
8,Basic,Household budget cuts,54
9,Basic,Password sharing crackdown,38


### 4.26 Cancellation Reasons Across Subscription Plans

**Business Question:** What are the most common cancellation reasons across different subscription plans?

**Key Findings:**

- **Basic Plan:** Price is the leading cancellation reason, with 201 customers. Other common reasons include alternative platforms (111) and insufficient content (105).
- **Standard Plan:** Price is the leading reason (115), followed by alternative platforms (107) and customers no longer needing the service (89).
- **Premium Plan:** Switching to an alternative platform is the leading reason (66), followed by price (49) and customers no longer needing the service (38).

**Interpretation:**

Price is the most common cancellation reason for Basic and Standard customers. Premium customers most frequently report switching to another platform. This suggests that cancellation factors may differ depending on the subscription plan.

**Business Recommendations:**
- Review pricing and value offered by Basic and Standard plans.
- Improve content variety and quality across subscription plans.
- Strengthen Premium plan benefits to remain competitive.
- Use cancellation feedback to develop plan-specific retention strategies.

In [38]:
query = """
WITH latest_payment AS (
    SELECT
        customer_id,
        payment_method,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY payment_date DESC, payment_id DESC
        ) AS rn
    FROM payments
    WHERE payment_method IS NOT NULL
)
SELECT
    lp.payment_method,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM latest_payment lp
JOIN churn_labels cl
    ON lp.customer_id = cl.customer_id
WHERE lp.rn = 1
GROUP BY lp.payment_method
ORDER BY total_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,payment_method,total_customers,churned_customers,churn_rate_percentage
0,Credit Card,2361,596,25.24
1,Debit Card,1419,357,25.16
2,UPI,1268,333,26.26
3,PayPal,1017,252,24.78
4,Net Banking,868,206,23.73
5,Mobile Wallet,597,130,21.78
6,Gift Card,470,117,24.89


### 4.27 Viewing Engagement Across Subscription Plans

**Business Question:** How does customer viewing engagement differ across subscription plans?

**Key Findings:**

| Subscription Plan | Customers | Average Sessions | Average Total Watch Minutes | Average Minutes per Session |
|---|---:|---:|---:|---:|
| Basic | 3,368 | 8.40 | 414.73 | 49.44 |
| Standard | 3,018 | 8.29 | 408.45 | 49.11 |
| Premium | 1,614 | 8.12 | 405.19 | 49.97 |

**Interpretation:**

Basic plan customers have the highest average total viewing time (414.73 minutes) and the highest average number of sessions (8.40). Premium customers have the highest average viewing time per session (49.97 minutes). However, the differences across plans are relatively small.

**Business Recommendations:**
- Monitor viewing engagement across all subscription plans.
- Study which content encourages customers to watch more frequently.
- Use viewing behaviour to recommend relevant content.
- Evaluate whether Premium plan features provide enough additional value to customers.

In [37]:
query = """
WITH customer_payment_methods AS (
    SELECT
        customer_id,
        GROUP_CONCAT(DISTINCT payment_method) AS payment_methods
    FROM payments
    WHERE payment_method IS NOT NULL
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN cpm.payment_methods IS NULL THEN 'No Payment Record'
        ELSE cpm.payment_methods
    END AS payment_method,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(100.0 * SUM(cl.churned) / COUNT(*), 2) AS churn_rate_percentage
FROM customers c
JOIN churn_labels cl
    ON c.customer_id = cl.customer_id
LEFT JOIN customer_payment_methods cpm
    ON c.customer_id = cpm.customer_id
GROUP BY payment_method
ORDER BY total_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,payment_method,total_customers,churned_customers,churn_rate_percentage
0,Credit Card,24,7,29.17
1,Debit Card,22,4,18.18
2,"Net Banking,Credit Card,UPI,Debit Card,PayPal",14,3,21.43
3,"Credit Card,UPI,Debit Card,PayPal,Net Banking",14,3,21.43
4,"Credit Card,Debit Card,Net Banking,PayPal,UPI",14,3,21.43
...,...,...,...,...
4466,"Credit Card,Debit Card,Gift Card,Mobile Wallet...",1,0,0.00
4467,"Credit Card,Debit Card,Gift Card,Mobile Wallet...",1,1,100.00
4468,"Credit Card,Debit Card,Gift Card,Mobile Wallet...",1,0,0.00
4469,"Credit Card,Debit Card,Gift Card,Mobile Wallet",1,0,0.00


### 4.28 Relationship Between Payment Method and Customer Churn

**Business Question:** Does customer churn differ according to the payment methods used?

**Key Findings:**

| Payment Method | Customers | Churned Customers | Churn Rate |
|---|---:|---:|---:|
| Credit Card | 2,361 | 596 | 25.24% |
| Debit Card | 1,419 | 357 | 25.16% |
| UPI | 1,268 | 333 | 26.26% |
| PayPal | 1,017 | 252 | 24.78% |
| Net Banking | 868 | 206 | 23.73% |
| Mobile Wallet | 597 | 130 | 21.78% |
| Gift Card | 470 | 117 | 24.89% |

**Interpretation:**

UPI users have the highest churn rate at 26.26%, while Mobile Wallet users have the lowest at 21.78%. However, the differences are relatively small. Payment method alone does not establish why customers churn.

**Business Recommendations:**
- Monitor payment failures and transaction issues across payment methods.
- Make payment options reliable and convenient.
- Investigate whether payment difficulties contribute to cancellations.
- Use payment behaviour alongside customer feedback and subscription history when identifying churn risk.

In [39]:
query = """
WITH latest_subscription AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
),
customer_ratings AS (
    SELECT
        customer_id,
        AVG(rating) AS average_rating
    FROM customer_feedback
    GROUP BY customer_id
)
SELECT
    sp.plan_name,
    COUNT(cr.customer_id) AS customers_with_feedback,
    ROUND(AVG(cr.average_rating), 2) AS average_customer_rating
FROM latest_subscription ls
JOIN subscription_plans sp
    ON ls.plan_id = sp.plan_id
JOIN customer_ratings cr
    ON ls.customer_id = cr.customer_id
WHERE ls.rn = 1
GROUP BY sp.plan_name
ORDER BY average_customer_rating DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,plan_name,customers_with_feedback,average_customer_rating
0,Standard,1355,3.21
1,Basic,1489,3.17
2,Premium,717,3.11


### 4.29 Customer Satisfaction Ratings Across Subscription Plans

**Business Question:** How do customer satisfaction ratings differ across subscription plans?

**Key Findings:**

| Subscription Plan | Customers with Feedback | Average Rating |
|---|---:|---:|
| Standard | 1,355 | 3.21 |
| Basic | 1,489 | 3.17 |
| Premium | 717 | 3.11 |

**Interpretation:**

The Standard plan has the highest average customer rating at 3.21, followed by Basic at 3.17 and Premium at 3.11. The differences are small, suggesting that customer satisfaction is relatively similar across the three plans.

**Business Recommendations:**
- Review feedback from Premium customers to understand areas for improvement.
- Continue monitoring customer satisfaction across all plans.
- Improve plan features and customer experience based on feedback.
- Use customer ratings alongside churn data to identify possible retention issues.

**Note:** Ratings are averaged at the customer level first, so customers with multiple feedback records do not receive extra weight simply because they submitted more feedback.

In [40]:
query = """
SELECT
    COALESCE(failed_payment_reason, 'Not Specified') AS failure_reason,
    COUNT(*) AS failed_transactions,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM payments WHERE payment_status = 'Failed'),
        2
    ) AS percentage_of_failed_transactions
FROM payments
WHERE payment_status = 'Failed'
GROUP BY failure_reason
ORDER BY failed_transactions DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,failure_reason,failed_transactions,percentage_of_failed_transactions
0,Bank declined,1278,17.74
1,Card expired,1211,16.81
2,Incorrect card details,1198,16.63
3,Insufficient funds,1189,16.50
4,Card reported lost/stolen,1180,16.38
5,International transaction blocked,1150,15.96


### 4.30 Analysis of Payment Failure Reasons

**Business Question:** What are the most common reasons for failed payment transactions?

**Key Findings:**

| Payment Failure Reason | Failed Transactions | Percentage |
|---|---:|---:|
| Bank declined | 1,278 | 17.74% |
| Card expired | 1,211 | 16.81% |
| Incorrect card details | 1,198 | 16.63% |
| Insufficient funds | 1,189 | 16.50% |
| Card reported lost/stolen | 1,180 | 16.38% |
| International transaction blocked | 1,150 | 15.96% |

**Interpretation:**

Bank declines are the most common payment failure reason, accounting for 17.74% of failed transactions. The other reasons have relatively similar percentages, indicating that payment failures occur due to several different issues rather than one dominant cause.

**Business Recommendations:**
- Notify customers when their cards are expired or payment details are incorrect.
- Provide alternative payment methods when a transaction fails.
- Help customers understand bank-related payment declines.
- Monitor payment failures and identify recurring transaction problems.

In [41]:
query = """
SELECT
    priority,
    COUNT(*) AS total_tickets,
    ROUND(AVG(resolution_time_hours), 2) AS average_resolution_hours,
    ROUND(MIN(resolution_time_hours), 2) AS minimum_resolution_hours,
    ROUND(MAX(resolution_time_hours), 2) AS maximum_resolution_hours
FROM support_tickets
WHERE resolution_time_hours IS NOT NULL
GROUP BY priority
ORDER BY
    CASE
        WHEN LOWER(priority) = 'high' THEN 1
        WHEN LOWER(priority) = 'medium' THEN 2
        WHEN LOWER(priority) = 'low' THEN 3
        ELSE 4
    END;
"""

result = pd.read_sql_query(query, conn)
display(result)

,priority,total_tickets,average_resolution_hours,minimum_resolution_hours,maximum_resolution_hours
0,High,1019,17.68,0.5,151.1
1,Medium,1565,17.91,0.5,131.0
2,Low,1629,17.96,0.5,126.4
3,Critical,379,19.37,0.5,136.7


### 4.31 Support Ticket Resolution Time by Priority

**Business Question:** How does support ticket resolution time differ by priority?

**Key Findings:**

| Priority | Total Tickets | Average Resolution Time | Minimum | Maximum |
|---|---:|---:|---:|---:|
| High | 1,019 | 17.68 hours | 0.5 hours | 151.1 hours |
| Medium | 1,565 | 17.91 hours | 0.5 hours | 131.0 hours |
| Low | 1,629 | 17.96 hours | 0.5 hours | 126.4 hours |
| Critical | 379 | 19.37 hours | 0.5 hours | 136.7 hours |

**Interpretation:**

Critical tickets have the longest average resolution time at 19.37 hours, while High-priority tickets have the shortest at 17.68 hours. The difference suggests that critical issues may require more complex investigation or additional support resources.

**Business Recommendations:**
- Review the workflow for critical support tickets.
- Prioritize urgent issues and monitor their resolution times.
- Identify reasons for unusually long-running tickets.
- Improve support processes to reduce resolution delays.

In [42]:
query = """
SELECT
    sentiment_label,
    COUNT(*) AS total_feedback,
    ROUND(AVG(rating), 2) AS average_rating,
    MIN(rating) AS minimum_rating,
    MAX(rating) AS maximum_rating
FROM customer_feedback
WHERE rating IS NOT NULL
GROUP BY sentiment_label
ORDER BY average_rating DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,sentiment_label,total_feedback,average_rating,minimum_rating,maximum_rating
0,Positive,1978,4.40,1,5
1,Neutral,1606,3.03,1,5
2,Negative,1462,1.64,1,5


### 4.32 Customer Ratings Across Feedback Sentiments

**Business Question:** How do customer ratings differ across positive, neutral, and negative feedback?

**Key Findings:**

| Sentiment | Total Feedback | Average Rating | Minimum Rating | Maximum Rating |
|---|---:|---:|---:|---:|
| Positive | 1,978 | 4.40 | 1 | 5 |
| Neutral | 1,606 | 3.03 | 1 | 5 |
| Negative | 1,462 | 1.64 | 1 | 5 |

**Interpretation:**

Positive feedback has the highest average rating of 4.40, while negative feedback has the lowest average rating of 1.64. Neutral feedback has an average rating of 3.03. These results show a clear relationship between feedback sentiment and customer ratings.

**Business Recommendations:**
- Review negative feedback to identify customer problems.
- Address recurring complaints about content, payments, and technical issues.
- Continue improving the customer experience.
- Monitor sentiment and ratings to measure customer satisfaction over time.

In [43]:
query = """
SELECT
    issue_category,
    COUNT(*) AS total_tickets,
    ROUND(AVG(customer_satisfaction_score), 2) AS average_satisfaction_score
FROM support_tickets
WHERE customer_satisfaction_score IS NOT NULL
GROUP BY issue_category
ORDER BY average_satisfaction_score ASC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,issue_category,total_tickets,average_satisfaction_score
0,Billing & Payments,1287,2.98
1,Streaming Quality,1000,3.04
2,Account Access,751,3.47
3,Subscription,662,3.50
4,Content,337,3.55
5,Device & App,555,3.56


### 4.33 Customer Satisfaction by Support Issue Category

**Business Question:** Which support issue categories receive the lowest customer satisfaction ratings?

**Key Findings:**

| Support Issue Category | Total Tickets | Average Satisfaction Score |
|---|---:|---:|
| Billing & Payments | 1,287 | 2.98 |
| Streaming Quality | 1,000 | 3.04 |
| Account Access | 751 | 3.47 |
| Subscription | 662 | 3.50 |
| Content | 337 | 3.55 |
| Device & App | 555 | 3.56 |

**Interpretation:**

Billing & Payments has the lowest average satisfaction score (2.98), followed by Streaming Quality (3.04). Device & App issues receive the highest average score (3.56). This suggests that billing and streaming problems are important areas for customer support improvement.

**Business Recommendations:**
- Improve billing and payment support processes.
- Investigate recurring streaming quality complaints.
- Provide clear troubleshooting guidance for technical issues.
- Monitor satisfaction scores after support improvements.

In [46]:
query = """
SELECT
    COALESCE(c.gender, 'Not Specified') AS gender,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(
        100.0 * SUM(cl.churned) / COUNT(*), 2
    ) AS churn_rate_percentage
FROM customers c
JOIN churn_labels cl
    ON c.customer_id = cl.customer_id
GROUP BY gender
ORDER BY total_customers DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,gender,total_customers,churned_customers,churn_rate_percentage
0,Female,3817,967,25.33
1,Male,3779,931,24.64
2,Other,247,50,20.24
3,Prefer not to say,157,43,27.39


### 4.34 Customer Churn Analysis by Gender

**Business Question:** Does customer churn vary across gender groups?

**Key Findings:**

| Gender | Total Customers | Churned Customers | Churn Rate |
|---|---:|---:|---:|
| Female | 3,817 | 967 | 25.33% |
| Male | 3,779 | 931 | 24.64% |
| Other | 247 | 50 | 20.24% |
| Prefer not to say | 157 | 43 | 27.39% |

**Interpretation:**

Customers who selected "Prefer not to say" have the highest observed churn rate at 27.39%, while the Other category has the lowest at 20.24%. Female and male churn rates are relatively close. The smaller group sizes should be considered when interpreting these differences.

**Business Recommendations:**
- Focus retention strategies on customer behaviour rather than gender alone.
- Investigate common churn reasons across customer groups.
- Use additional factors such as viewing engagement, payment history, and feedback to understand churn.
- Respect customer privacy and preferences when using demographic information.

In [47]:
query = """
WITH latest_subscription AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY subscription_start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
),
payment_summary AS (
    SELECT
        customer_id,
        MAX(CASE WHEN payment_status = 'Failed' THEN 1 ELSE 0 END) AS has_failed_payment
    FROM payments
    GROUP BY customer_id
),
viewing_summary AS (
    SELECT
        customer_id,
        SUM(watch_duration_minutes) AS total_watch_minutes
    FROM viewing_activity
    GROUP BY customer_id
),
support_summary AS (
    SELECT
        customer_id,
        AVG(customer_satisfaction_score) AS avg_support_satisfaction
    FROM support_tickets
    WHERE customer_satisfaction_score IS NOT NULL
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN ls.auto_renew = 1 THEN 'Auto-Renew Enabled'
        ELSE 'Auto-Renew Disabled'
    END AS customer_factor,
    COUNT(*) AS total_customers,
    SUM(cl.churned) AS churned_customers,
    ROUND(100.0 * SUM(cl.churned) / COUNT(*), 2) AS churn_rate
FROM latest_subscription ls
JOIN churn_labels cl
    ON ls.customer_id = cl.customer_id
WHERE ls.rn = 1
GROUP BY customer_factor

UNION ALL

SELECT
    CASE
        WHEN ps.has_failed_payment = 1 THEN 'Failed Payment'
        ELSE 'No Failed Payment'
    END,
    COUNT(*),
    SUM(cl.churned),
    ROUND(100.0 * SUM(cl.churned) / COUNT(*), 2)
FROM customers c
JOIN churn_labels cl ON c.customer_id = cl.customer_id
LEFT JOIN payment_summary ps ON c.customer_id = ps.customer_id
GROUP BY 1

UNION ALL

SELECT
    CASE
        WHEN vs.total_watch_minutes < 500 THEN 'Low Viewing'
        WHEN vs.total_watch_minutes < 1500 THEN 'Medium Viewing'
        ELSE 'High Viewing'
    END,
    COUNT(*),
    SUM(cl.churned),
    ROUND(100.0 * SUM(cl.churned) / COUNT(*), 2)
FROM customers c
JOIN churn_labels cl ON c.customer_id = cl.customer_id
LEFT JOIN viewing_summary vs ON c.customer_id = vs.customer_id
GROUP BY 1

UNION ALL

SELECT
    CASE
        WHEN ss.avg_support_satisfaction < 2.5 THEN 'Low Support Satisfaction'
        WHEN ss.avg_support_satisfaction < 3.5 THEN 'Medium Support Satisfaction'
        ELSE 'High Support Satisfaction'
    END,
    COUNT(*),
    SUM(cl.churned),
    ROUND(100.0 * SUM(cl.churned) / COUNT(*), 2)
FROM customers c
JOIN churn_labels cl ON c.customer_id = cl.customer_id
LEFT JOIN support_summary ss ON c.customer_id = ss.customer_id
WHERE ss.avg_support_satisfaction IS NOT NULL
GROUP BY 1

ORDER BY customer_factor;
"""

result = pd.read_sql_query(query, conn)
display(result)

,customer_factor,total_customers,churned_customers,churn_rate
0,Auto-Renew Disabled,1470,422,28.71
1,Auto-Renew Enabled,6530,1569,24.03
2,Failed Payment,4533,1287,28.39
3,High Support Satisfaction,1536,286,18.62
4,High Viewing,28,10,35.71
5,Low Support Satisfaction,784,262,33.42
6,Low Viewing,5645,1474,26.11
7,Medium Support Satisfaction,1220,331,27.13
8,Medium Viewing,2327,507,21.79
9,No Failed Payment,3467,704,20.31


### 4.35 Combined Analysis of Customer Churn Risk Indicators

**Business Question:** Which customer factors show the strongest differences in churn rates?

**Key Findings:**

| Customer Factor | Total Customers | Churned Customers | Churn Rate |
|---|---:|---:|---:|
| Auto-Renew Disabled | 1,470 | 422 | 28.71% |
| Auto-Renew Enabled | 6,530 | 1,569 | 24.03% |
| Failed Payment | 4,533 | 1,287 | 28.39% |
| No Failed Payment | 3,467 | 704 | 20.31% |
| Low Support Satisfaction | 784 | 262 | 33.42% |
| Medium Support Satisfaction | 1,220 | 331 | 27.13% |
| High Support Satisfaction | 1,536 | 286 | 18.62% |
| Low Viewing | 5,645 | 1,474 | 26.11% |
| Medium Viewing | 2,327 | 507 | 21.79% |
| High Viewing | 28 | 10 | 35.71% |

**Interpretation:**

The results show noticeable differences in churn rates across several customer factors.

- Customers with failed payments have a higher churn rate (28.39%) than those without failed payments (20.31%).
- Customers with auto-renew disabled have a higher churn rate (28.71%) than those with auto-renew enabled (24.03%).
- Customers with low support satisfaction have a higher churn rate (33.42%) than those with high satisfaction (18.62%).
- Low-viewing customers have a churn rate of 26.11%. The high-viewing group contains only 28 customers, so its 35.71% churn rate should be interpreted cautiously.

**Business Recommendations:**
- Improve payment reliability and help customers resolve failed transactions.
- Encourage customers to use auto-renewal where appropriate.
- Improve support quality and resolve recurring customer complaints.
- Identify customers with declining viewing engagement.
- Combine multiple customer indicators when developing churn prevention strategies.

**Important Note:** These results show associations, not proof that any single factor causes customer churn. The high-viewing group is very small and should not be used for strong conclusions.

## 5. Overall Key Findings

The SQL analysis produced the following important business insights:

1. **Customer Churn:** Overall customer churn is 24.89%, representing 1,991 out of 8,000 customers.
2. **Subscription Plans:** Basic is the most popular plan, accounting for 42.10% of customers.
3. **Pricing:** Price is a leading cancellation reason, particularly among Basic and Standard subscribers.
4. **Payment Issues:** Customers with failed payments have a higher churn rate (28.39%) than customers without failed payments (20.31%).
5. **Auto-Renewal:** Customers with auto-renewal disabled have a higher churn rate (28.71%) than those with it enabled (24.03%).
6. **Customer Support:** Customers with low support satisfaction have a churn rate of 33.42%, compared with 18.62% among customers with high support satisfaction.
7. **Customer Feedback:** Negative feedback has a lower average rating (1.64) than positive feedback (4.40).
8. **Viewing Engagement:** Viewing activity differs across engagement groups, although the high-viewing group is too small for strong conclusions.
9. **Content Preferences:** Movies account for the largest number of viewing sessions, while genre and language preferences vary across customers.
10. **Payment Failures:** Bank declines are the most common recorded payment failure reason, accounting for 17.74% of failed transactions.

These findings highlight the importance of pricing, payment reliability, customer support, and viewing engagement in understanding customer behaviour.

## 6. Business Recommendations

Based on the SQL analysis, the following actions are recommended:

- **Improve Customer Retention:** Identify customers showing signs of cancellation and provide relevant retention offers.
- **Review Subscription Pricing:** Evaluate plan pricing and benefits, especially for Basic and Standard subscribers.
- **Improve Payment Reliability:** Help customers resolve failed transactions and provide alternative payment options.
- **Strengthen Customer Support:** Focus on billing and streaming quality issues, which received relatively low satisfaction scores.
- **Improve Content Strategy:** Use viewing patterns and customer preferences to guide content recommendations.
- **Monitor Customer Feedback:** Regularly analyze ratings and sentiment to identify customer concerns.
- **Encourage Engagement:** Recommend relevant content to customers with lower viewing activity.
- **Use Data-Driven Decisions:** Combine subscription, payment, viewing, and support information to identify customers who may be at risk of churn.

These recommendations can help improve customer experience, support retention, and guide business decisions.

## 7. Conclusion

The SQL Business Analysis phase explored Netflix customer, subscription, payment, viewing, support, content, and feedback datasets using SQLite and SQL queries.

The analysis identified important patterns related to customer churn, subscription preferences, payment failures, viewing engagement, and customer satisfaction.

The findings indicate that payment problems, low support satisfaction, and subscription preferences are useful indicators to investigate when understanding customer churn.

This phase provided business insights that will support the next stages of the project, including feature engineering, churn prediction, customer segmentation, customer lifetime value analysis, and dashboard development.